# Shell command — guide opérationnel

Ce notebook regroupe les commandes utiles pour préparer, compiler, installer au démarrage et nettoyer un exécutable Studiovision Autosync.

Les exemples sont volontairement génériques. Remplacer les valeurs entre chevrons par les chemins et noms réellement utilisés sur le poste cible.

Convention utilisée dans ce notebook :

- `<PROJECT_DIR>` : dossier racine du projet ;
- `<SCRIPT_PATH>` : chemin du script Python à compiler ;
- `<APP_NAME>` : nom de l’exécutable à générer ;
- `<DIST_DIR>` : dossier contenant l’exécutable généré ;
- `<EXE_PATH>` : chemin complet de l’exécutable ;
- `<SHORTCUT_NAME>` : nom du raccourci Windows de démarrage.

## 1. Se placer dans le dossier du projet

Toujours lancer les commandes depuis la racine du projet. Cela évite les erreurs de chemin avec `requirements.txt`, `src/`, `build/` et `dist/`.

In [ ]:
cd "<PROJECT_DIR>"

## 2. Créer un environnement Python isolé

Cette étape est recommandée pour éviter de mélanger les dépendances du projet avec celles d’autres programmes Python installés sur le poste.

In [ ]:
python -m venv .venv
.venv\Scripts\activate
python -m pip install --upgrade pip

## 3. Installer les dépendances

Les bibliothèques nécessaires sont listées dans `requirements.txt`. L’installation doit être faite dans l’environnement Python activé.

In [ ]:
pip install -r requirements.txt

## 4. Tester le script en mode Python

Avant de générer un exécutable, vérifier que le script démarre correctement en Python. Cette étape permet de détecter rapidement les erreurs de configuration des chemins, des dépendances ou de l’accès à StudioVision.

In [ ]:
python "<SCRIPT_PATH>"

## 5. Générer l’exécutable avec PyInstaller

La commande suivante produit un exécutable autonome dans le dossier `dist/`. Le nom de l’application doit rester cohérent avec le nom du script source.

L’option `--noconsole` masque la console Windows. Pour une phase de diagnostic, il peut être utile de retirer temporairement cette option afin de voir les logs directement à l’écran.

In [ ]:
pyinstaller --onefile --noconsole --name "<APP_NAME>" "<SCRIPT_PATH>"

## 6. Vérifier le résultat de compilation

Après compilation, contrôler que l’exécutable existe bien dans `dist/`.

In [ ]:
dir dist

## 7. Définir les variables de déploiement Windows

Les commandes suivantes sont écrites en PowerShell. Elles définissent les chemins utilisés pour créer un raccourci dans le dossier de démarrage de l’utilisateur Windows.

In [ ]:
$AppName = "<APP_NAME>"
$ExePath = "<EXE_PATH>"
$DistDir = "<DIST_DIR>"
$ShortcutName = "<SHORTCUT_NAME>.lnk"
$StartupDir = [System.Environment]::GetFolderPath("Startup")

## 8. Créer le raccourci de démarrage automatique

Cette commande crée un raccourci `.lnk` dans le dossier Startup de l’utilisateur courant. Au prochain démarrage de session, Windows lancera l’exécutable automatiquement.

Utiliser de préférence une version du programme avec verrou mono-instance pour éviter les doubles lancements.

In [ ]:
$Shell = New-Object -ComObject WScript.Shell
$Shortcut = $Shell.CreateShortcut((Join-Path $StartupDir $ShortcutName))
$Shortcut.TargetPath = $ExePath
$Shortcut.WorkingDirectory = $DistDir
$Shortcut.Save()

## 9. Vérifier le dossier de démarrage

Cette commande permet de contrôler que le raccourci a bien été créé.

In [ ]:
dir $StartupDir

## 10. Lancer l’exécutable manuellement pour test

Avant de redémarrer le poste, lancer une fois l’exécutable manuellement afin de vérifier les logs, l’accès aux dossiers et le comportement avec StudioVision.

In [ ]:
& "<EXE_PATH>"

## 11. Arrêter un processus existant

Cette commande est utile avant de remplacer un exécutable déjà lancé. Remplacer `<PROCESS_NAME>` par le nom du processus sans extension `.exe`.

In [ ]:
Stop-Process -Name "<PROCESS_NAME>" -Force -ErrorAction SilentlyContinue

## 12. Supprimer un raccourci de démarrage

À utiliser lorsqu’un programme ne doit plus se lancer automatiquement à l’ouverture de session.

In [ ]:
Remove-Item (Join-Path $StartupDir $ShortcutName) -Force -ErrorAction SilentlyContinue

## 13. Supprimer une ancienne tâche planifiée si elle existe

Si une ancienne installation utilisait le Planificateur de tâches Windows, cette commande permet de retirer la tâche. Elle n’est pas nécessaire si le démarrage automatique se fait uniquement par raccourci Startup.

In [ ]:
schtasks /delete /tn "<TASK_NAME>" /f

## 14. Nettoyer les fichiers de build

Après compilation, PyInstaller crée des dossiers temporaires. Ils peuvent être supprimés pour repartir d’un état propre avant une nouvelle compilation.

In [ ]:
rmdir /s /q build
rmdir /s /q dist
del "<APP_NAME>.spec"

## 15. Checklist de validation

Avant mise en production, vérifier les points suivants :

- l’exécutable démarre sans erreur ;
- le dossier source existe et reçoit bien les images ;
- le dossier d’orphelins existe ou peut être créé ;
- `PUBLIC.MDB` est accessible ;
- StudioVision est ouvert sur le bon patient pendant le test ;
- l’image est déplacée dans le bon dossier patient ;
- le document apparaît dans StudioVision ;
- les logs ne contiennent pas d’erreur bloquante ;
- une seule instance du programme est active.